# Experimentando com bancos de dados em Python

## Conectando com a base de dados

Existem sistemas de gerenciamento de bancos de dados que operam no mesmo processo da aplicação cliente - ou seja, são bibliotecas que você pode importar diretamente no seu programa. Por exemplo: a biblioteca ``sqlite3`` (https://docs.python.org/3/library/sqlite3.html) é parte do conjunto de bibliotecas padrão da linguagem Python, e permite criar bases de dados diretamente em arquivo ou memória, sem a necessidade de um programa em separado. Eis um exemplo de uso de sqlite3:

In [19]:
import sqlite3

db = sqlite3.connect("teste.db")
# Para criar a base em memória (base temporária) use o nome de arquivo
# ":memory:"

Note que o arquivo ``teste.db`` foi criado. 

In [20]:
import os

print("teste.db" in os.listdir())

True


Apesar do ``sqlite3`` criar o sistema de gerenciamento de banco de dados (SGBD) SQLite diretamente no nosso processo (neste caso, o kernel Python que está sendo executado por esse notebook), o objeto de acesso ainda é uma "conexão". Isso se deve ao fato de que SGBDs "normais" - aqueles que realmente usamos em produção - são geralmente *sistemas cliente-servidor*, onde o SGBD é um processo separado da aplicação cliente e a comunicação entre eles acontece através de uma conexão de rede.

<center><img src='imgs/client-server.png'></center>

**Atividade:**

Na figura acima, quem é o cliente do sistema de gerenciamento de banco de dados?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

O cliente é a aplicação que solicita operações ao SGBD. Neste caso, o notebook Python é o cliente e o servidor MySQL é o SGBD. O MySQL Workbench também é outro cliente do mesmo servidor.

</div>

Uma conexão é simplesmente o conduíte entre o cliente e o SGBD. Em Python, a especificação PEP 249 (https://www.python.org/dev/peps/pep-0249/) define que uma conexão deve ter os métodos ``.close()`` (para fechar a conexão), ``.cursor()`` (para obter um *cursor*, que é o objeto para realmente interagir com o banco de dados), e o par ``.commit()``/``.rollback()`` (para gerenciamento de transações).

Vamos fechar essa conexão e trabalhar com o MySQL daqui em diante.

In [21]:
db.close()

In [22]:
os.remove("teste.db")

Antes de tudo vamos criar um banco de dados. Queremos criar um banco de dados para gerenciar as equipes do primeiro campeonato Insper de ninja-cowboy-urso!

- Ninja, com sua velocidade e furtividade, detona o cowboy.
- O cowboy atira no urso.
- O urso dá uma patada no ninja e acaba com ele.

Cada jogador tem um nome, uma equipe, e sua jogada favorita. As equipes tem um nome e um grito de guerra.

Portanto, nosso schema será:

```
jogador (id PK, nome_equipe, nome, preferencia)
equipe (nome PK, grito)
```

com a restrição:

```
jogador(nome_equipe) FK para equipe(nome)
```

E um dicionário de dados simples para este problema é:

<center><b>jogador</b></center>
<table>
    <tr>
        <th>Campo</th>
        <th>Tipo</th>
        <th>Significado</th>
        <th>Chave?</th>
        <th>Valores</th>
    </tr>
    <tr>
        <td>id</td>
        <td>int</td>
        <td>Identificador unico do jogador</td>
        <td>PK</td>
        <td>sem restrições</td>
    </tr>
    <tr>
        <td>nome_equipe</td>
        <td>string</td>
        <td>Nome da equipe na qual o jogador atua</td>
        <td>FK equipe(nome)</td>
        <td></td>
    </tr>
    <tr>
        <td>nome</td>
        <td>string</td>
        <td>Nome do jogador</td>
        <td></td>
        <td>Pode haver dois jogadores com mesmo nome. Máximo 80 caracteres.</td>
    </tr>
    <tr>
        <td>preferencia</td>
        <td>int</td>
        <td>Jogada preferida do jogador</td>
        <td></td>
        <td>0: Ninja, 1: Cowboy, 2: Urso</td>
    </tr>
</table>

<center><b>equipe</b></center>
<table>
    <tr>
        <th>Campo</th>
        <th>Tipo</th>
        <th>Significado</th>
        <th>Chave?</th>
        <th>Valores</th>
    </tr>
    <tr>
        <td>nome</td>
        <td>string</td>
        <td>Nome da equipe</td>
        <td>PK</td>
        <td>Máximo 30 caracteres</td>
    </tr>
    <tr>
        <td>grito</td>
        <td>string</td>
        <td>Grito de guerra da equipe</td>
        <td></td>
        <td>Maximo 80 caracteres</td>
    </tr>
</table>

**Atividade:** Escreva e rode o script de criação do banco de dados. Chame o banco de dados de `torneio`. Compare com o **gabarito** em `sql/script_001.sql`.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Execute, no **Workbench** o script para criação da base `torneio` disponível em `sql/script_001.sql`.

</div>

Para conectar com o MySQL precisamos da biblioteca *mysql-connector-python*.
 
Para instalar esta biblioteca (já está nos `requirements.txt`, provavelmente não precisará instalar):

**`!python -m pip install mysql-connector-python==8.3.0`**

In [23]:
from dotenv import load_dotenv
import insperautograder.jupyter as ia
import mysql.connector
import os

In [24]:
print(f"mysql.connector.__version__: {mysql.connector.__version__}")

mysql.connector.__version__: 8.3.0


In [25]:
load_dotenv(override=True)

True

Vamos agora abrir uma conexão com o MySQL. Como agora temos realmente um sistema cliente-servidor, precisamos definir onde está o servidor, e precisamos também de informações de autenticação.

In [26]:
connection_options = {
    "host": "localhost",
    "user": "root",
    "password": "123456",
    "database": "torneio"
}

**Pergunta**: Faz sentido guardar o *user* e *password* dentro do próprio código fonte? Qual o jeito correto de lidar com username e password no seu produto?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Não. Usuário e senha são informações sensíveis e não devem ficar no código-fonte ou no Git. Em desenvolvimento, podem ser lidos de variáveis de ambiente ou de um arquivo `.env` ignorado pelo Git; em produção, o ideal é usar um gerenciador de segredos e permissões mínimas.

</div>

**Resposta**: 

Não se deve guardar *user* e *password* dentro do código fonte, pois trata-se de:
- Informação sensível e que não deve ser acessada pelos programadores.
- Informação que muda com frequência - ou pelo menos deveria mudar!
- Dado puro, que deve ser separado do código.

Algumas alternativas populares para resolver esse problema são:
- Use um arquivo de configuração na conta que será usada para rodar a aplicação. Este arquivo deve ter todas as permissões de acesso negadas, exceto permissão de leitura para o próprio usuário da conta de execução da aplicação. O arquivo deverá ser lido então pela aplicação no momento da inicialização.
- Use uma variável de ambiente com essas credenciais. É a mesma coisa que no ítem anterior, só ao invés de ter um arquivo de configuração da aplicação, temos o arquivo de configuração da conta em si, com os mesmos requisitos em relação às restrições de permissão.

## Atualizando `.env`

### Tarefa:

Atualize o arquivo `.env`. Por enquanto ele provavelmente contém apenas as variáveis criadas para a API de correção automática:

```python
IAG_SERVER_URL="https://bigdata.insper-comp.com.br/iag"
IAG_SUBJECT="megadados"
IAG_OFFERING="xxxx"
IAG_TOKEN="iagtok_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
```

Vamos adicionar três novas variáveis:

- `MD_DB_SERVER`: o endereço para o host onde o mysql está instalado (provavelmente `"localhost"`)
- `MD_DB_USERNAME`: o usuário para acesso ao banco (provavelmente `"root"`)
- `MD_DB_PASSWORD`: a senha do usuário.


O `.env` ficará neste formato (veja as três novas variáveis ao final):

```python
IAG_SERVER_URL="https://bigdata.xxxxxxx"
IAG_SUBJECT="alguma_disciplina"
IAG_OFFERING="xxx"
IAG_TOKEN="iagtok_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
MD_DB_SERVER="localhost"
MD_DB_USERNAME="root"
MD_DB_PASSWORD="alguma_senha"
```

In [27]:
load_dotenv(override=True)

connection_options = {
    "host": os.getenv("MD_DB_SERVER"),
    "user": os.getenv("MD_DB_USERNAME"),
    "password": os.getenv("MD_DB_PASSWORD"),
    "database": "torneio"
}

**Abrindo a conexão:**

In [34]:
# A notação de duplo asterisco em Python permite usar um dicionário 
# no lugar de argumentos nomeados. A linha a ser executada abaixo equivale á:
#
# connection = mysql.connector.connect(
#     host=os.getenv("MD_DB_SERVER"),
#     user=os.getenv("MD_DB_USERNAME"),
#     password=os.getenv("MD_DB_PASSWORD"),
#     database="torneio"
# )

connection_options = {
    "host": os.getenv("MD_DB_SERVER"),
    "user": os.getenv("MD_DB_USERNAME"),
    "password": os.getenv("MD_DB_PASSWORD"),
    "database": "torneio"
}

connection = mysql.connector.connect(**connection_options)

ProgrammingError: 1045 (28000): Access denied for user 'root'@'172.17.0.1' (using password: YES)

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**IMPORTANTE:**

TODAS AS OPERAÇÕES REALIZADAS NESTA CONEXÃO SÃO TEMPORÁRIAS, ATÉ QUE O COMANDO `connection.commit()` seja executado!

</div>

<div class="alert alert-warning" role="alert">
    
**MAIS IMPORTANTE AINDA:**

ISSO NÃO VALE SE **AUTOCOMMIT** ESTIVER LIGADO!

</div>


Estudaremos melhor essa questão quando discutirmos transações em maiores detalhes.

Vamos criar alguns dados iniciais:

In [35]:
T1 = "Raposas Nerds"
T2 = "É nois"

NINJA = 0
COWBOY = 1
URSO = 2

times = {
    T1: "sudo vencer!",
    T2: "Olha eu mamãe!",
}

jogadores = [
    ("Raul Ayres", T1, NINJA),
    ("Luciano Hashimoto", T1, COWBOY),
    ("Rafael Montaigner", T1, URSO),
    ("Igor Miranda", T2, URSO),
    ("Andrew Ikeda", T2, COWBOY),
    ("Fábio Kurauchi", T2, NINJA),
]

Vamos inserir os dados dos times.

**Atividade:** Quem devemos inserir primeiro: dados na tabela `equipe` ou na tabela `jogador`?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Devemos inserir primeiro em `equipe`, que é a tabela pai. `jogador` é a tabela filha e sua chave estrangeira `nome_equipe` somente pode referenciar uma equipe que já exista.

</div>

Quando uma tabela tem chave estrangeira para outra tabela dizemos que ela é filha (*child*) da outra tabela, e que a outra tabela é a pai (*parent*).

Um *cursor* é o objeto que permite executar queries SQL e interagir com os resultados obtidos.

### Tentando inserir um jogador

Vamos tentar inserir um jogador (vai falhar):

In [36]:
try:
    with connection.cursor() as cursor:
        cursor.execute("INSERT INTO jogador (nome, nome_equipe, preferencia) VALUES ('Joao', 'sete e meia', 0)")
    connection.commit()
except mysql.connector.IntegrityError as exception:
    print(f"IntegrityError: {exception}")
    connection.rollback()

NameError: name 'connection' is not defined

Perceba que obtemos uma **exceção**, pois estamos tentando utilizar uma equipe inexistente na tabela de `equipe`.

Isto responde nossa pergunta anterior: precisamos inserir primeiro as equipes.

Vamos fazer isto utilizando um laço que fará todas as inserções:

In [37]:
try:
    with connection.cursor() as cursor:
        # Outra opção aqui seria o uso de cursor.executemany()
        for time, grito in times.items():
            cursor.execute("INSERT INTO equipe VALUES (%s, %s)", (time, grito))
    connection.commit()
except mysql.connector.IntegrityError as exception:
    print(f"IntegrityError: {exception}")
    connection.rollback()

NameError: name 'connection' is not defined

Note o comando ``commit`` acima: ele serve para efetivar a operação no banco de dados. Verifique no MySQL Workbench que a operação foi bem sucedida.

**Atividade:** Perceba que desta vez não especificamos os nomes das colunas no `INSERT`. É necessário deixar claro os campos da tabela?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Não é obrigatório quando fornecemos valores para todas as colunas exatamente na ordem da tabela. Mesmo assim, informar as colunas é recomendado porque deixa o comando claro e evita erros se a estrutura ou a ordem das colunas mudar.

</div>

Podemos tambem usar nossa conexão para verificar o estado da tabela ``equipe``:

In [38]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM equipe")
    for item in cursor:
        print(item)

NameError: name 'connection' is not defined

Podemos usar também nosso velho conhecido `pandas`!

In [39]:
import pandas as pd
pd.read_sql_query("SELECT * FROM equipe", connection)

NameError: name 'connection' is not defined

**Atividade:** Tente executar novamente a inserção de dados, o que acontece?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

A segunda inserção falha com erro de chave primária duplicada, pois `equipe.nome` é a PK e os nomes `Raposas Nerds` e `É nois` já estão cadastrados.

</div>

## Interlúdio: sanitização de entradas de dados

Por que usamos aqueles ``%s`` no comando a ser executado? Por que não simplesmente incluir diretamente os dados na *string* do comando SQL? Por exemplo:

```Python
cursor.execute("INSERT INTO equipe VALUES ('" + time + "', '" + grito + "')")
```

Muitas vezes o resultado é o mesmo! Contudo, e SE executássemos a query abaixo...

time = "Robert', ''); DROP DATABASE torneio; -- "
grito = ""

with connection.cursor() as cursor:
    for result in cursor.execute(
            "INSERT INTO equipe VALUES ('" + time + "', '" + grito + "')",
            multi=True,
    ):
        print(result)
    connection.commit()

with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM equipe")
    for item in cursor:
        print(item)

connection.close()

A célula acima está desabilitada. Mas se quiser executar passe para tipo "Code" e rode. Depois, garanta que a conexão está fechada com `connection.close()`, volte para o começo do notebook e rode tudo de novo exceto a célula mortal!

![Little Bobby Tables](imgs/exploits_of_a_mom.png)

[Exploits of a mom](https://xkcd.com/327/)


Existem jeitos de quebrar o sistema com esse tipo de ataque, onde o hacker tenta inserir código SQL mal-intencionado em formulários, na expectativa de que o programador não tenha feito a sanitização das entradas. Esse ataque chama-se ***'SQL injection'***.

Fim do interlúdio, voltamos à apresentação principal.

Vamos agora inserir os jogadores. Neste caso não precisamos inserir a chave primária, pois ela é gerada automaticamente. Consequentemente temos que especificar que estamos inserindo apenas as demais colunas - temos que nomear as colunas na qual estamos inserindo os dados.

In [ ]:
try:
    with connection.cursor() as cursor:
        cursor.executemany(
            "INSERT INTO jogador (nome, nome_equipe, preferencia) VALUES (%s, %s, %s)",
            jogadores)
    connection.commit()
except Exception as e:
    print(e)
    connection.rollback()

Vamos verificar se funcionou. Para evitar ter criar os cursores a todo momento e coletar os resultados vamos construir uma classe auxiliar, só para simplificar o código:

In [ ]:
class ConnectionHelper:

    def __init__(self, connection):
        self.connection = connection

    def __call__(self, query, args=None):
        with self.connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

In [ ]:
db = ConnectionHelper(connection)

Note o padrão de *'dependency injection'* aqui: ao invés de criar a conexão dentro do construtor da classe ConnectionHelper, passamos um objeto 'connection' que será armazenado dentro do ConnectionHelper. Qualquer objeto 'connection' pode ser passado, de qualquer classe, desde que obedeça a interface (implícita) de um objeto de conexão ao banco de dados. Neste nosso caso, tem que ter o método `cursor()` que retorne um objeto de interação com a base de dados.

## Explorando a estrutura da base de dados

Vamos ver quais tabelas existem na base ``torneio``:

In [40]:
db("SHOW TABLES")

ProgrammingError: Cannot operate on a closed database.

Para saber qual o schema da tabela `jogador`, podemos usar o comando '`DESCRIBE`'

In [41]:
db("DESCRIBE jogador")

ProgrammingError: Cannot operate on a closed database.

## Consultando a base de dados

Vamos usar o comando `SELECT` para listar os conteudos da tabela 'Jogador'

In [ ]:
db("SELECT * FROM jogador")

O comando acima lista todos os registros da tabela `jogador`, com todas as colunas presentes:

![Seleção da tabela inteira](imgs/tudo.png)

Vamos agora selecionar apenas algumas colunas para exibir.

In [ ]:
db("SELECT nome, nome_equipe FROM jogador")

Agora vemos apenas as colunas escolhidas.  A operação de seleção de colunas chama-se **projeção**.

![Projeção](imgs/projecao.png)

Vamos agora atuar na escolha de linhas, selecionando quais desejamos. Para escolher todas as linhas cujo `nome_equipe` começa com 'Rap' podemos executar a query a seguir:

In [ ]:
db("SELECT * FROM jogador WHERE nome_equipe LIKE 'Rap%'")

Vemos apenas as linhas escolhidas. A operação de filtragem de linhas apropriadas chama-se ... **seleção**!!!

![Seleção](imgs/selecao.png)

Seleção e projeção são termos advindos da *álgebra relacional*, tópico que discutiremos em aulas futuras.

O comando `SELECT` também pode ser usado para efetuar cálculos:

In [42]:
db("SELECT 1 + 1")

ProgrammingError: Cannot operate on a closed database.

## Onde foram parar os dados?

MySQL mantém um grande conjunto de variáveis globais para que possamos verificar o estado do sistema. Podemos consultar essas variáveis usando SQL! Afinal, se queremos armazenar informação (neste caso, sobre o próprio sistema), nada melhor que um SGBD!

In [ ]:
db("SHOW VARIABLES")

Para descobrir o valor de uma variável específica, use a cláusula `WHERE`:

In [ ]:
db("SHOW VARIABLES WHERE Variable_name = 'version'")

Para listar todas as variáveis com "dir" no nome:

In [ ]:
db("SHOW VARIABLES WHERE Variable_name LIKE '%dir%'")

Observe a variável `datadir`: ela contém o diretório onde o MySQL armazena nossos dados. Por exemplo no Windows: `C:\ProgramData\MySQL\MySQL Server 8.0\Data\`

<center><img src='imgs/datadir.png'/></center>

Note que esse diretório também tem várias chaves criptográficas! Uma vez que o acesso físico ao seu servidor for comprometido, você pode perder completamente o controle, pois o invasor agora tem até mesmo as chaves de acesso!

Dentro do diretório `torneio` temos nossa base de dados no disco:

<center><img src='imgs/torneio.png'/></center>


## Carregando dados em massa

Se você tem um arquivo com vários itens de dados, o melhor jeito de inserí-los no banco de dados é usar os comandos de carga de dados do seu SGBD preferido. No MySQL, veja a documentação do comando `LOAD DATA` em https://dev.mysql.com/doc/refman/8.0/en/load-data.html

Daqui para frente acostume-se a consultar a documentação de todo novo comando que você descobrir!

# `UPDATE`

Vamos alterar informações na nossa base. Para isso vamos usar o comando `UPDATE`. 

Suponha que desejamos alterar o grito de guerra da equipe "Raposas Nerds" de 'sudo vencer!' para 'sudo vencer --force!':

In [ ]:
db("UPDATE equipe SET grito='sudo vencer --force!' WHERE nome='Raposas Nerds'")

Verificando o resultado:

In [ ]:
db("SELECT * from equipe")

**Atividade:** Passe o "Rafael Montaigner" para o time "É nois".

In [ ]:
db("SELECT * FROM jogador")

In [ ]:
db("UPDATE jogador SET nome_equipe = 'É nois' WHERE nome = 'Rafael Montaigner'")

In [ ]:
db("SELECT * FROM jogador")

**Atividade:** Tente mudar o nome de "Andrew Ikeda" para "Andrew Gomes da Silva". O que aconteceu?

In [ ]:
db("UPDATE jogador SET nome = 'Andrew Gomes da Silva' WHERE nome = 'Andrew Ikeda'")

In [ ]:
db("SELECT * FROM jogador")

**Atividade:** Tente agora mudar o nome do time "É nois" para "Somos nos". O que aconteceu?

In [ ]:
db("UPDATE equipe SET nome = 'Somos nos' WHERE nome = 'É nois'")

In [ ]:
db("SELECT * FROM equipe")

## `COMMIT` e `ROLLBACK`

Note que as ultimas atividades (de `UPDATE`) foram feitas sem executar `connection.commit()`. Neste caso todas as modificações realizadas ainda não foram registradas no banco de dados! Estas mudanças existem apenas na nossa *sessão*. Vamos verificar esse fenômeno. Primeiro vamos ver o estado das nossas tabelas:

In [ ]:
db("SELECT * FROM jogador")

In [ ]:
db("SELECT * FROM equipe")

Agora vamos verificar o estado do banco de dados lá no MySQL Workbench:

<center><img src='imgs/equipes.png'/></center>

<center><img src='imgs/jogadores.png'/></center>

Está diferente! Isso acontece porque as mudanças da nossa sessão ainda não foram *committed*, não foram registradas em definitivo. Para registrar as mudanças você pode executar o comando `connection.commit()`, ou equivalentemente rodar o comando SQL `COMMIT`. Se você se arrependeu das mudanças realizadas na sessão, jogue elas fora com o comando `connection.rollback()` ou equivalentemente o comando SQL `ROLLBACK`.

Vamos então concretizar nossas mudanças:

In [ ]:
connection.commit()
# ou então db("COMMIT"), é a mesma coisa.

Agora vamos verificar lá no MySQL Workbench:

<center><img src='imgs/equipes2.png'/></center>

<center><img src='imgs/jogadores2.png'/></center>

Podemos constatar que as mudanças feitas na nossa sessão agora são visíveis na outra sessão (a do MySQL Workbench). Vamos revisitar futuramente os comandos `COMMIT` e `ROLLBACK` na aula de transações SQL.

## `DELETE`

Vamos remover o 'Fábio Kurauchi' da tabela `jogador`

In [43]:
try:
    db("DELETE FROM jogador WHERE nome='Fábio Kurauchi'")
    connection.commit()
except Exception as e:
    print(f"{type(e)}: {e}")
    connection.rollback()

<class 'sqlite3.ProgrammingError'>: Cannot operate on a closed database.


NameError: name 'connection' is not defined

Verificando o resultado:

In [ ]:
db("SELECT * FROM jogador")

**Atividade:** Remova da tabela de jogadores todos aqueles que preferem jogar cowboy.

In [ ]:
db("DELETE FROM jogador WHERE preferencia = 1")

In [ ]:
db("SELECT * FROM jogador")

Agora vamos tentar remover o time 'É nois':

In [ ]:
try:
    db("DELETE FROM equipe WHERE nome='É nois'")
    connection.commit()
except Exception as e:
    print(f"{type(e)}: {e}")
    connection.rollback()

Novamente o malfadado erro `IntegrityError`. Isso acontece porque o SGBD está fazendo exatamente o que você pediu: garantindo a integridade dos dados! Afinal, se removermos a equipe `'É nois'`, o que fazemos com as linhas da tabela-filha que referenciam esse registro?

A restrição de chave estrangeira no nome da equipe (em `script_001.sql` ela foi chamada de `fk_equipe`) especifica, implicitamente, que a ação de apagar um registro deve ser bloqueada se impactar tabelas filhas. Podemos mudar essa restrição.

## `ON UPDATE / ON DELETE`

Vamos reescrever a restrição de chave estrangeira. Copie o script a seguir para `script_002.sql`.

```mysql
USE torneio;

ALTER TABLE jogador
    DROP FOREIGN KEY fk_equipe;

ALTER TABLE jogador
    ADD CONSTRAINT fk_equipe FOREIGN KEY (nome_equipe) REFERENCES equipe (nome)
    ON DELETE CASCADE
    ON UPDATE CASCADE;

```

Feche a conexão atual com o banco de dados: ela está bloqueando alterações no schema. Não se esqueça de efetivar as modificações passadas com `commit()`, se elas não foram *committed* até o momento:

In [ ]:
connection.commit()
connection.close()

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Agora rode `script_002.sql` no MySQL Workbench. Se tudo deu certo você deve observar a mudança de ação na restrição de chave estrageira:

</div>

<br>
<center><img src='imgs/cascade.png'/></center>
<br>

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Reabra a conexão e recrie o *helper*:

</div>

In [ ]:
connection = mysql.connector.connect(**connection_options)
db = ConnectionHelper(connection)

Vamos agora tentar mudar o nome da equipe 'É nois' para 'Somos nós':

In [ ]:
db("UPDATE equipe SET nome='Somos nós' WHERE nome='É nois'")

Verificando o resultado:

In [ ]:
db("SELECT * from equipe")

Vamos verificar o que aconteceu com a tabela `jogador`

In [ ]:
db("SELECT * FROM jogador")

Como você pode ver, a mudança de nome foi devidamente propagada, cortesia do `ON UPDATE CASCADE`.

Já o `ON DELETE CASCADE` é mais bruto. Vamos remover o time 'Somos nós':

In [ ]:
db("DELETE FROM equipe WHERE nome='Somos nós'")

In [ ]:
db("SELECT * from equipe")

Ok. E os jogadores?

In [ ]:
db("SELECT * FROM jogador")

A ação `CASCADE` associada ao evento `ON DELETE` acabou por limpar a tabela `jogador` tambem.

Vamos cancelar todas essas ações com o comando `rollback()`, que é o oposto de `commit()`

In [ ]:
connection.rollback()

Verificando se deu certo:

In [ ]:
db("SELECT * FROM jogador")

**Atividade:** Alem de `CASCADE` e `RESTRICT` existem outros especificadores de ação. Ache a URL da documentação que mostra essas opções.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

A documentação está em https://dev.mysql.com/doc/refman/8.0/en/create-table-foreign-keys.html. As ações são `RESTRICT`, `CASCADE`, `SET NULL`, `NO ACTION` e `SET DEFAULT`; no InnoDB, `NO ACTION` equivale a `RESTRICT` e `SET DEFAULT` não é aceito.

</div>

## `REPLACE` = `DELETE` + `INSERT`

O comando `REPLACE` lembra um pouco o comando `UPDATE`, e é aí que mora o perigo! `REPLACE` é na verdade uma combinação de `DELETE` seguido de `INSERT`. Vamos exemplificar: suponha que eu quero atualizar o grito do time 'É nois' para 'Vai dar ruim!', mas ao invés de usar o `UPDATE` resolvi usar o `REPLACE`:

In [ ]:
db("SELECT * from equipe")
db("SELECT * from jogador")

In [ ]:
db("REPLACE INTO equipe VALUES ('É nois', 'Vai dar ruim!')")

In [ ]:
db("SELECT * from equipe")

In [ ]:
db("SELECT * from jogador")

Aparentemente deu ruim!

**Atividade:** Explique

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

`REPLACE` não atualiza a linha existente como `UPDATE`. Ao encontrar a mesma chave primária, ele remove a equipe antiga e insere uma nova. A remoção ativa `ON DELETE CASCADE`, apagando os jogadores associados; a nova equipe é inserida depois, mas os jogadores não são recriados.

</div>

Ufa, terminamos a primeira parte! Feche a conexão e siga para a ATV!

In [ ]:
connection.rollback()
connection.close()

## Atividade para prática de DML

Faça a atividade para prática de DML. Vamos importar e redefinir o  `ConnectionHelper` para facilitar a vida!

In [44]:
# Importando tudo para facilitar caso precise reiniciar!
import os
import insperautograder.jupyter as ia
import mysql.connector
from mysql.connector import IntegrityError, ProgrammingError
from dotenv import load_dotenv

load_dotenv(override=True)

class ConnectionHelper:

    def __init__(self, connection):
        self.connection = connection

    def __call__(self, query, args=None):
        with self.connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

### Conferindo a API de Autograding

#### Tarefas e Notas
Vamos conferir as tarefas e notas

In [45]:
ia.tasks()

|    | Atividade   | De                  | Até                 | Conta como ATV?   | % Nota Atraso   |
|---:|:------------|:--------------------|:--------------------|:------------------|:----------------|
|  0 | newborn     | 2026-08-10 00:00:00 | 2026-11-30 00:00:00 | Não               | 0%              |
|  1 | select01    | 2026-08-14 00:00:00 | 2026-08-19 23:59:59 | Sim               | 25%             |
|  2 | ddl         | 2026-08-26 00:00:00 | 2026-09-01 23:59:59 | Sim               | 25%             |
|  3 | dml         | 2026-08-28 00:00:00 | 2026-09-01 23:59:59 | Sim               | 25%             |
|  4 | agg_join    | 2026-09-02 00:00:00 | 2026-09-08 23:59:59 | Sim               | 25%             |

In [46]:
ia.grades(task="dml")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | dml         | ex01        |      1 |     10 |                10 |                 0 |
|  1 | dml         | ex02        |      1 |     10 |                10 |                 0 |
|  2 | dml         | ex03        |      1 |     10 |                10 |                 0 |
|  3 | dml         | ex04        |      1 |     10 |                10 |                 0 |
|  4 | dml         | ex05        |      1 |     10 |                10 |                 0 |
|  5 | dml         | ex06        |      1 |     10 |                10 |                 0 |

In [47]:
ia.grades(by="TASK")

|    | Tarefa   |   Nota | Conta como ATV?   |
|---:|:---------|-------:|:------------------|
|  0 | newborn  |   10   | Não               |
|  1 | select01 |    2.5 | Sim               |
|  2 | ddl      |   10   | Sim               |
|  3 | dml      |   10   | Sim               |
|  4 | agg_join |    0   | Sim               |

Calcular sua média de atividades:


In [48]:
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |             10 |

### Criar Base de Dados

Vamos começar pela criação da base de dados. Ela deverá se chamar `vendinha`.

<img src="imgs/vendinha.png">

<p></p>

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Execute no workbench o script `vendinha.sql` disponibilizado pelo professor.

</div>

In [50]:
connection_options_atv = {
    "host": os.getenv("MD_DB_SERVER"),
    "user": os.getenv("MD_DB_USERNAME"),
    "password": os.getenv("MD_DB_PASSWORD"),
    "database": "vendinha"
}

connection_atv = mysql.connector.connect(**connection_options_atv)

db_atv = ConnectionHelper(connection_atv)

ProgrammingError: 1045 (28000): Access denied for user 'root'@'172.17.0.1' (using password: YES)

**Dica**: Caso necessite executar códigos SQL no Workbench, libere antes a conexão.

```python
connection_atv.commit() #ou connection_atv.rollback() 
connection_atv.close()
```

**Exercício 1**: Faça uma query para inserir a região `"N"` com descrição `"Norte"` na tabela `regiao`.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**DICA:**
    
Agora que temos a função `db` para conexão com o banco, podemos testar nossas queries aqui mesmo no notebook, alternando comandos de **DML** com `SELECT` para conferir o resultado.
</div>

In [ ]:
# Deixa eu ver como está
db_atv("SELECT * FROM regiao")

In [ ]:
# Agora deixa eu responder a questão
sql_ex01 = """
INSERT INTO regiao (regiao, descricao)
VALUES ('N', 'Norte');
"""

db_atv(sql_ex01)

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM regiao")

In [ ]:
ia.sender(answer="sql_ex01", task="dml", question="ex01", answer_type="pyvar")

**Exercício 2**: Em uma única query (sem fazer vários `INSERT`), insira as regiões na tabela `regiao`:

- a região `"NE"` com descrição `"Nordeste"`.
- a região `"SE"` com descrição `"Sudeste"`.
- a região `"S"` com descrição `"Sul"`.
- a região `"CO"` com descrição `"Centro-Oeste"`.

In [51]:
# Deixa eu ver como está
db_atv("SELECT * FROM regiao")

NameError: name 'db_atv' is not defined

In [52]:
# Agora deixa eu responder a questão
sql_ex02 = """
INSERT INTO regiao (regiao, descricao)
VALUES
    ('NE', 'Nordeste'),
    ('SE', 'Sudeste'),
    ('S', 'Sul'),
    ('CO', 'Centro-Oeste');
"""

db_atv(sql_ex02)

NameError: name 'db_atv' is not defined

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM regiao")

In [53]:
ia.sender(answer="sql_ex02", task="dml", question="ex02", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex02', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 3**: Considere que as seguinte querie foi executada:

```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('sp', 'São Paulo', 'SE'),
    ('mg', 'Minas Gerais', 'SE'),
    ('pr', 'Paraná', 'S'),
    ('am', 'Amazonas', 'N'),
    ('ba', 'Bahia', 'NE');
```

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Crie uma que altere para **maiúsculo** o `uf` apenas dos estados da região **sudeste**.

In [54]:
# Preparando o banco
sql_prep = """INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('sp', 'São Paulo', 'SE'),
    ('mg', 'Minas Gerais', 'SE'),
    ('pr', 'Paraná', 'S'),
    ('am', 'Amazonas', 'N'),
    ('ba', 'Bahia', 'NE');
"""

db_atv(sql_prep)

NameError: name 'db_atv' is not defined

In [ ]:
# Deixa eu ver como está
db_atv("SELECT * FROM uf")

In [ ]:
# Agora deixa eu responder a questão
sql_ex03 = """
UPDATE uf
SET uf = UPPER(uf)
WHERE regiao = 'SE';
"""

db_atv(sql_ex03)

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM uf")

In [ ]:
ia.sender(answer="sql_ex03", task="dml", question="ex03", answer_type="pyvar")

**Exercício 4**: Considere que as seguinte queries foram executadas:

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Perceba que os `uf` estão em maúsculo neste exercício!

</div>


**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Foi descoberto que todos os vendedores cadastrados em **agosto de 2023** são bots e devem ser removidos da base. Construa sua query!

In [ ]:
# Preparando o banco
# Tenta remover todas as referências aos vendedores!
try:
    db_atv("TRUNCATE vendedor_vende_cidade")
except ProgrammingError as e:
    print("Este código foi criado para que consiga executar este exercício novamente\napós ter avançado para os próximos!")
    print(f"Pode ignorar este erro, provavelmente a tabela ainda nao existe!\nErro: {e}")

In [ ]:
# Preparando o banco
# Tenta remover todas as referências aos vendedores!
try:
    db_atv("TRUNCATE vendedor_ativo_dia")
except ProgrammingError as e:
    print("Este código foi criado para que consiga executar este exercício novamente\napós ter avançado para os próximos!")
    print(f"Pode ignorar este erro, provavelmente a tabela ainda nao existe!\nErro: {e}")

In [ ]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

In [ ]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor")

In [ ]:
# Agora deixa eu responder a questão
sql_ex04 = """
DELETE FROM vendedor
WHERE data_cad >= '2023-08-01'
  AND data_cad < '2023-09-01';
"""

db_atv(sql_ex04)

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor")

In [ ]:
ia.sender(answer="sql_ex04", task="dml", question="ex04", answer_type="pyvar")

**Exercício 5**: Considere que as seguinte queries foram executadas:

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Perceba que os `uf` estão em maúsculo neste exercício!

</div>

**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```


**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Dadas as relações de **vendedores** e **cidades onde vendedor vende**:

- Bruna Fontana:
    - São Paulo SP
    - Campinas SP
- Silvio Jardim:
    - São Paulo SP
    - Belo Horizonte MG
    - São Roque de Minas
- Camila Oliveira:
    - Manaus AM
    
Construa uma query para popular a tabela `vendedor_vende_cidade` (Veja Diagrama do Modelo Relacional).

In [ ]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

In [ ]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor_vende_cidade")

In [ ]:
# Agora deixa eu responder a questão
sql_ex05 = """
INSERT INTO vendedor_vende_cidade (id_vendedor, id_cidade)
VALUES
    (103, 20),
    (103, 21),
    (102, 20),
    (102, 24),
    (102, 25),
    (106, 23);
"""

db_atv(sql_ex05)

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor_vende_cidade")

In [ ]:
ia.sender(answer="sql_ex05", task="dml", question="ex05", answer_type="pyvar")

**Exercício 6**: Considere que as seguinte queries foram executadas:

**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```

Ainda, considere uma nova tabela chamada `vendedor_ativo_dia` com o esquema:

| Coluna         | Tipo         | PK (Primary Key?) | Not Null? | Autoinc?
|----------------|--------------|-------------------|-----------|-----------|
| id             | INT          |   True            |    True   |     True  |
| id_vendedor    | INT          |                   |    True   |           |
| ativo          | TYNYINT      |                   |    True   |           |
| data_registro  | DATE         |                   |    True   |           |

Considere que a tabela `vendedor_ativo_dia` é utilizada para criar um registro histórico de todos os vendedores que estão ativos em cada data.

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Construa uma query que faça a inserção dos dados na tabela `vendedor_ativo_dia` a partir de um `SELECT` dos vendedores ativos da tabela de vendedor.

<div class="alert alert-danger">

Não deixe a data **hardcoded**. Pesquise como recuperar a data atual do sistema em uma query.

</div>
  
<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Por simplificação, não é necessário criar as constraints de chave extrangeira na tabela `vendedor_ativo_dia`.

</div>

Sugestão de resolução:
- Criar DDL para `vendedor_ativo_dia`
- Criar SELECT para pegar:
    - Vendedores ativos
    - com o dia de hoje
- Inserir estes dados (usando `SELECT`) na tabela `vendedor_ativo_dia`.

Não é necessário enviar a **DDL** para o servidor.

In [ ]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

In [ ]:
ddl_vendedor_ativo_dia="""
DROP TABLE IF EXISTS vendedor_ativo_dia;

CREATE TABLE vendedor_ativo_dia (
    id INT NOT NULL AUTO_INCREMENT,
    id_vendedor INT NOT NULL,
    ativo TINYINT NOT NULL,
    data_registro DATE NOT NULL,
    PRIMARY KEY (id)
);
"""

db_atv(ddl_vendedor_ativo_dia)

In [ ]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor_ativo_dia")

In [ ]:
# Agora deixa eu responder a questão (INSERT)
sql_ex06 = """
INSERT INTO vendedor_ativo_dia (id_vendedor, ativo, data_registro)
SELECT id, ativo, CURDATE()
FROM vendedor
WHERE ativo = 1;
"""

db_atv(sql_ex06)

In [ ]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor_ativo_dia")

In [ ]:
ia.sender(answer="sql_ex06", task="dml", question="ex06", answer_type="pyvar")

### Conferir Notas

Confira se as notas na atividade são as esperadas!

Primeiro na atividade atual!

In [14]:
ia.grades(by="TASK", task="dml")

|    | Tarefa   |   Nota | Conta como ATV?   |
|---:|:---------|-------:|:------------------|
|  0 | dml      |     10 | Sim               |

In [15]:
ia.grades(task="dml")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | dml         | ex01        |      1 |     10 |                10 |                 0 |
|  1 | dml         | ex02        |      1 |     10 |                10 |                 0 |
|  2 | dml         | ex03        |      1 |     10 |                10 |                 0 |
|  3 | dml         | ex04        |      1 |     10 |                10 |                 0 |
|  4 | dml         | ex05        |      1 |     10 |                10 |                 0 |
|  5 | dml         | ex06        |      1 |     10 |                10 |                 0 |

E agora em todas as demais!

In [16]:
ia.grades(by="TASK")

|    | Tarefa   |   Nota | Conta como ATV?   |
|---:|:---------|-------:|:------------------|
|  0 | newborn  |   10   | Não               |
|  1 | select01 |    2.5 | Sim               |
|  2 | ddl      |   10   | Sim               |
|  3 | dml      |   10   | Sim               |

In [17]:
ia.grades()

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | newborn     | ex01        |      1 |   10   |                10 |               0   |
|  1 | select01    | ex01        |      1 |    2.5 |                 0 |               2.5 |
|  2 | select01    | ex02        |      1 |    2.5 |                 0 |               2.5 |
|  3 | select01    | ex03        |      1 |    2.5 |                 0 |               2.5 |
|  4 | select01    | ex04        |      1 |    2.5 |                 0 |               2.5 |
|  5 | select01    | ex05        |      1 |    2.5 |                 0 |               2.5 |
|  6 | ddl         | ex02        |      1 |   10   |                10 |               0   |
|  7 | ddl         | ex03        |      1 |   10   |                10 |               0   |
|  8 | ddl         | ex04        |      1 |   10   |                10 |               0   |
|  9 | ddl         | ex05        |      1 |   10   |                10 |               0   |
| 10 | ddl         | ex06        |      1 |   10   |                10 |               0   |
| 11 | ddl         | ex07        |      1 |   10   |                10 |               0   |
| 12 | ddl         | ex09        |      1 |   10   |                10 |               0   |
| 13 | ddl         | ex10        |      1 |   10   |                10 |               0   |
| 14 | ddl         | ex11        |      1 |   10   |                10 |               0   |
| 15 | dml         | ex01        |      1 |   10   |                10 |               0   |
| 16 | dml         | ex02        |      1 |   10   |                10 |               0   |
| 17 | dml         | ex03        |      1 |   10   |                10 |               0   |
| 18 | dml         | ex04        |      1 |   10   |                10 |               0   |
| 19 | dml         | ex05        |      1 |   10   |                10 |               0   |
| 20 | dml         | ex06        |      1 |   10   |                10 |               0   |
| 21 | agg_join    | ex01        |      1 |    0   |                 0 |               0   |
| 22 | agg_join    | ex02        |      1 |    0   |                 0 |               0   |
| 23 | agg_join    | ex03        |      1 |    0   |                 0 |               0   |
| 24 | agg_join    | ex04        |      1 |    0   |                 0 |               0   |
| 25 | agg_join    | ex05        |      1 |    0   |                 0 |               0   |
| 26 | agg_join    | ex06        |      1 |    0   |                 0 |               0   |

Calcular sua média de atividades:


In [18]:
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |             10 |